# AI Quant-Mate Session 2 — 지능형 포트폴리오 비중 관리

주제: **"내 포트는 정말 분산되어 있는가?"**

흐름:
1. portfolio.csv 준비
2. CSV 로드
3. 평가금액 계산
4. 비중 계산
5. 섹터별 집계 (groupby)
6. HHI 집중도 지표
7. 리포트 출력
8. AI 비평 (OpenAI)

> 셀은 위에서 아래로 순서대로 실행한다.

## Step 0 — 라이브러리 설치 & import

In [ ]:
!pip install pandas openai -q

In [ ]:
import pandas as pd
from openai import OpenAI

---
## Step 1 — 실습용 portfolio.csv 생성

본인 포트폴리오 CSV가 이미 있으면 이 셀은 건너뛰고 Step 2에서 경로만 바꿔도 된다.

컬럼 스펙: `ticker, name, quantity, avg_price, current_price, sector`

In [ ]:
sample_rows = [
    {"ticker": "NVDA", "name": "NVIDIA",    "quantity": 10, "avg_price": 800, "current_price": 950, "sector": "AI/반도체"},
    {"ticker": "AMD",  "name": "AMD",       "quantity": 15, "avg_price": 120, "current_price": 140, "sector": "AI/반도체"},
    {"ticker": "TSM",  "name": "TSMC",      "quantity": 20, "avg_price": 100, "current_price": 110, "sector": "반도체"},
    {"ticker": "AAPL", "name": "Apple",     "quantity":  5, "avg_price": 170, "current_price": 190, "sector": "IT/기기"},
    {"ticker": "MSFT", "name": "Microsoft", "quantity":  5, "avg_price": 400, "current_price": 420, "sector": "IT/소프트웨어"},
]
pd.DataFrame(sample_rows).to_csv("portfolio.csv", index=False, encoding="utf-8-sig")
print("✅ portfolio.csv 생성 완료")

---
## Step 2 — CSV 로드

`pd.read_csv`로 파일을 읽어 `df`에 담는다. `head()`로 구조 확인.

In [ ]:
df = pd.read_csv("portfolio.csv")
print(f"행: {len(df)}개, 컬럼: {list(df.columns)}")
df.head()

---
## Step 3 — 평가금액 계산

`value = quantity × current_price`

pandas는 벡터 연산을 지원해 모든 행이 한 번에 계산된다.

In [ ]:
df["value"] = df["quantity"] * df["current_price"]
df[["name", "quantity", "current_price", "value"]]

---
## Step 4 — 총 자산 & 비중 계산

- `total_value`: 평가금액 합계
- `weight`: 각 종목이 전체에서 차지하는 비율 (합 = 1.0)

In [ ]:
total_value = df["value"].sum()
df["weight"] = df["value"] / total_value

print(f"총 자산: ${total_value:,.0f}")
print(f"비중 합: {df['weight'].sum():.4f} (1.0이어야 정상)")
df[["name", "value", "weight"]]

---
## Step 5 — 섹터별 집계 (groupby)

**Split → Apply → Combine**
- Split: `sector`별로 데이터를 쪼갠다
- Apply: 각 그룹의 `weight` 합을 구한다
- Combine: 하나의 표로 합친다

→ 개별 종목이 아니라 **"어떤 산업에 돈이 몰려있는가"**를 본다.

In [ ]:
sector_weights = (
    df.groupby("sector")["weight"]
      .sum()
      .reset_index()
      .sort_values(by="weight", ascending=False)
)
sector_weights

---
## Step 6 — HHI (집중도 지표)

$$HHI = \sum w_i^2$$

각 섹터 비중을 **제곱**해서 더한다. 제곱 때문에 큰 비중이 훨씬 부각되어 쏠림이 드러난다.

| HHI | 상태 |
|---|---|
| < 0.15 | 잘 분산 |
| 0.15 ~ 0.25 | 적당한 집중 |
| > 0.25 | 고위험 집중 |

In [ ]:
hhi = (sector_weights["weight"] ** 2).sum()

if hhi > 0.25:
    status = "⚠️ 고위험 (매우 집중됨)"
elif hhi > 0.15:
    status = "⚖️ 주의 (집중됨)"
else:
    status = "✅ 양호 (잘 분산됨)"

print(f"HHI: {hhi:.3f}  →  {status}")

---
## Step 7 — 리포트 출력

In [ ]:
print("="*40)
print("📊 포트폴리오 분석 리포트")
print("="*40)
print(f"💰 총 자산 가치: ${total_value:,.0f}")
print(f"🔍 섹터 집중도(HHI): {hhi:.3f}")
print(f"🛡️ 리스크 상태: {status}")
print("\n📈 상세 섹터 비중:")
for _, row in sector_weights.iterrows():
    print(f"- {row['sector']}: {row['weight']:.1%}")

---
## Step 8 — AI 비평용 데이터 구성

AI가 이해하기 좋은 dict 구조로 정리. 프롬프트 변수에 꽂아 쓴다.

In [ ]:
analysis_summary = {
    "sector_data": sector_weights.to_dict(orient="records"),
    "hhi_score": round(hhi, 3),
}
analysis_summary

---
## Step 9 — OpenAI 클라이언트

Colab이면 `userdata.get('OPENAI_API_KEY')`, 로컬이면 환경변수 사용.

In [ ]:
# --- Colab ---
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# --- 로컬 ---
import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)
print("✅ client 생성 완료")

---
## Step 10 — 프롬프트 작성

In [ ]:
SYSTEM_PROMPT = """
너는 월스트리트 15년차 헤지펀드 퀀트 애널리스트다.
사용자의 포트폴리오 섹터별 비중과 HHI 지수를 받아 리스크를 비평한다.
반드시 아래 형식으로만 답하라.

📊 현재 구조: (한 줄 요약)
🎯 핵심 리스크: (쏠림 섹터 또는 HHI 해석)
💡 리밸런싱 제안: (어떤 섹터 비중을 어떻게 조절할지 구체적으로)
⚠️ 모니터링 포인트: (앞으로 지켜볼 지표 한 줄)
"""

user_prompt = (
    f"내 포트폴리오의 섹터별 비중은 {analysis_summary['sector_data']}이고, "
    f"HHI 지수는 {analysis_summary['hhi_score']}다. "
    "이 포트가 특정 섹터에 쏠려 있는지 퀀트 관점에서 비평하고, "
    "리스크를 낮추기 위한 제안을 하라."
)
print(user_prompt)

---
## Step 11 — AI 호출 & 결과 출력

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ],
    max_tokens=500,
    temperature=0.3,
)
ai_critique = response.choices[0].message.content
print("🤖 AI 퀀트 비평")
print("="*40)
print(ai_critique)

---
## Step 12 — 결과 저장 (다음 주차 재료)

분석 결과를 CSV로 남겨 다음 주차에 다시 불러 쓴다.

In [ ]:
from datetime import datetime
stamp = datetime.today().strftime("%Y%m%d")

out_df = df[["ticker", "name", "sector", "value", "weight"]].copy()
out_df.to_csv(f"portfolio_weights_{stamp}.csv", index=False, encoding="utf-8-sig")
sector_weights.to_csv(f"sector_weights_{stamp}.csv", index=False, encoding="utf-8-sig")

with open(f"ai_critique_{stamp}.txt", "w", encoding="utf-8") as f:
    f.write(f"HHI: {hhi:.3f}\n상태: {status}\n\n{ai_critique}")

print(f"✅ 저장 완료: portfolio_weights_{stamp}.csv, sector_weights_{stamp}.csv, ai_critique_{stamp}.txt")